#  HUE 20-Year Energy Distribution 
Runs data discovery, source validation, 20-year augmentation, outage sensitivity, figures, audit, and ZIP export.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile
RUN_MODE = 'full'  # change to 'preliminary' for a 48-hour integration check
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
print('Mode:', RUN_MODE)

In [ ]:
# Locate an unpacked project; otherwise extract the uploaded project ZIP.
projects = list(INPUT.rglob('run_experiment.py'))
if projects:
    PROJECT = projects[0].parent
else:
    zips = [p for p in INPUT.rglob('*.zip') if 'Energy_Distribution' in p.name]
    if not zips:
        raise FileNotFoundError('Project input is missing: no run_experiment.py or project ZIP')
    extracted = WORK / 'energy_distribution_project'
    extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(zips[0]) as archive: archive.extractall(extracted)
    projects = list(extracted.rglob('run_experiment.py'))
    if not projects: raise FileNotFoundError('run_experiment.py missing inside project ZIP')
    PROJECT = projects[0].parent
solar = list(INPUT.rglob('Solar.csv'))
HUE = next((p.parent for p in solar if list(p.parent.glob('Residential_*.csv'))), None)
if HUE is None: raise FileNotFoundError('Attach the HUE Kaggle dataset first')
print('Project:', PROJECT)
print('HUE:', HUE)

In [ ]:
# Source-level checks.
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT, check=True)

In [ ]:
RESULTS = WORK / ('HUE_AUGMENTED_20Y_FINAL' if RUN_MODE == 'full' else 'HUE_PRELIMINARY_CHECK')
cmd = [sys.executable, 'run_experiment.py', '--hue-root', str(HUE),
       '--output', str(RESULTS), '--prosumers', '10', '--augment-years', '20',
       '--augmentation-start-year', '2021', '--validation-years', '2', '--test-years', '1',
       '--outage-scenario', 'moderate', '--full', '--save-augmented-data', '--resume']
if RUN_MODE == 'full':
    cmd += ['--episodes', '30', '--rl-seeds', '42,52,62', '--training-window-days', '90']
else:
    cmd += ['--episodes', '1', '--rl-seeds', '42,52,62', '--training-window-days', '2', '--max-test-hours', '48']
print(' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT, check=True)

In [ ]:
audit = [sys.executable, 'scripts/audit_outputs.py', str(RESULTS)]
if RUN_MODE != 'full': audit.append('--allow-preliminary')
subprocess.run(audit, cwd=PROJECT, check=True)
archive = shutil.make_archive(str(WORK / RESULTS.name), 'zip', RESULTS)
print('Downloadable results:', archive)

In [ ]:
import pandas as pd
display(pd.read_csv(RESULTS / 'RUN_MANIFEST.csv'))
display(pd.read_csv(RESULTS / 'tables/outage_sensitivity_summary.csv'))